In [ ]:
!nvidia-smi

Fri Sep 18 13:01:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

REPO_URL = "https://github.com/siifat/green-llm-token-research.git"
PROJECT = "/content/drive/MyDrive/green-llm-token-research"

if not os.path.exists(PROJECT):
    !git clone "$REPO_URL" "$PROJECT"
else:
    print("Project folder already exists.")

%cd "$PROJECT"

Project folder already exists.
/content/drive/MyDrive/green-llm-token-research


In [ ]:
from pathlib import Path
import csv

csv_path = Path("green_llm_ready_bundle/green_llm_ready_100_prompts.csv")

print("Dataset exists:", csv_path.exists())

with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

print("Prompt pairs:", len(rows))

p3 = [r for r in rows if r["prompt_id"] == "P0003"][0]
print("P0003 source pair:", p3["source_pair_id"])

Dataset exists: True
Prompt pairs: 100
P0003 source pair: 1.26


In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 93 not upgraded.


Install Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import os
import subprocess
import time

os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/ollama_models"
os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(5)
print("Ollama server started.")

Ollama server started.


In [ ]:
!ollama list

NAME           ID              SIZE      MODIFIED           
phi3.5:3.8b    61819fb370a3    2.2 GB    About a minute ago    


In [ ]:
!ollama pull phi3.5:3.8b

In [ ]:
!ollama run phi3.5:3.8b "Reply with exactly: READY"

READY

I'm Phi, an AI, so I don't have feelings, but I'm ready to assist you! What
What can I help you with?



In [ ]:
!ollama ps

NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
phi3.5:3.8b    61819fb370a3    3.8 GB    100% GPU     4096       4 minutes from now    


In [ ]:
from pathlib import Path

source = Path("scripts/run_experiment.py")
target = Path("scripts/run_experiment_phi35_colab.py")

text = source.read_text(encoding="utf-8")

text = text.replace(
    'MODEL = "gemma3:4b"',
    'MODEL = "phi3.5:3.8b"'
)

text = text.replace(
    'output_path = project_root / "data" / "raw" / "runs.jsonl"',
    'output_path = project_root / "data" / "raw" / "runs_phi35_38b_colab.jsonl"'
)

text = text.replace(
    'error_path = project_root / "logs" / "experiment_errors.jsonl"',
    'error_path = project_root / "logs" / "experiment_errors_phi35_38b_colab.jsonl"'
)

target.write_text(text, encoding="utf-8")

print("Created:", target)

Created: scripts/run_experiment_phi35_colab.py


In [ ]:
!grep -n 'MODEL =' scripts/run_experiment_phi35_colab.py

11:MODEL = "phi3.5:3.8b"


In [ ]:
!grep -n 'runs_phi35' scripts/run_experiment_phi35_colab.py

151:    output_path = project_root / "data" / "raw" / "runs_phi35_38b_colab.jsonl"


In [ ]:
!python scripts/run_experiment_phi35_colab.py --limit 1

Model: phi3.5:3.8b
Tasks in scope: 1
Expected runs: 2
Already completed: 1
Remaining: 1
Output: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_phi35_38b_colab.jsonl

Warming up the model...
Warm-up complete.

[1/1] P0001 - baseline ... 
    Ollama error on attempt 1/4. Retrying in 5s...

    Ollama error on attempt 2/4. Retrying in 10s...

    Ollama error on attempt 3/4. Retrying in 20s...
FAILED after retries
[1/1] P0001 - optimized ... SKIPPED (already complete)

Run pass finished.
Completed: 1/2
Still missing: 1

Missing runs:
  - P0001 baseline

Run the SAME command again later. The script will skip completed runs and retry only the missing ones.


In [ ]:
!tail -n 20 logs/experiment_errors_phi35_38b_colab.jsonl

{"timestamp_utc": "2026-09-18T13:08:33.749888+00:00", "prompt_id": "P0001", "variant": "baseline", "error_type": "RuntimeError", "error": "Ollama failed after 4 attempts: HTTP 500: Internal Server Error | Ollama message: {\"error\":\"prediction aborted, token repeat limit reached\"}"}
{"timestamp_utc": "2026-09-18T13:11:27.270906+00:00", "prompt_id": "P0001", "variant": "baseline", "error_type": "RuntimeError", "error": "Ollama failed after 4 attempts: HTTP 500: Internal Server Error | Ollama message: {\"error\":\"prediction aborted, token repeat limit reached\"}"}


In [ ]:
!tail -n 50 /tmp/ollama.log

slot print_timing: id  0 | task 9763 | n_gen =    577, tg =  63.85 t/s, tg_3s =  62.19 t/s
slot print_timing: id  0 | task 9763 | n_gen =    759, tg =  63.04 t/s, tg_3s =  60.61 t/s
slot print_timing: id  0 | task 9763 | n_gen =    937, tg =  62.29 t/s, tg_3s =  59.31 t/s
slot print_timing: id  0 | task 9763 | n_gen =   1110, tg =  61.49 t/s, tg_3s =  57.49 t/s
slot print_timing: id  0 | task 9763 | n_gen =   1278, tg =  60.70 t/s, tg_3s =  55.95 t/s
[GIN] 2026/09/18 - 13:10:41 | 500 | 24.027475142s |       127.0.0.1 | POST     "/api/generate"
srv          stop: cancel task, id_task = 9763
slot      release: id  0 | task 9763 | stop processing: n_tokens = 1540, truncated = 0
srv  update_slots: all slots are idle
srv  server_strea: conv_id= (empty=1)
slot get_availabl: id  0 | task -1 |  - checking sim = 1.000 (99/99) > 0.100
slot get_availabl: id  0 | task -1 | selected slot by LCP similarity, f_sim_best = 1.000 (> 0.100 thold), f_keep = 0.064
srv  get_availabl: updating prompt cache
s

In [ ]:
!ollama pull ministral-3:3b-instruct-2512-q4_K_M

In [ ]:
!ollama run ministral-3:3b-instruct-2512-q4_K_M "Reply with exactly: READY"

READY



In [ ]:
from pathlib import Path

source = Path("scripts/run_experiment.py")
target = Path("scripts/run_experiment_ministral3_colab.py")

text = source.read_text(encoding="utf-8")

text = text.replace(
    'MODEL = "gemma3:4b"',
    'MODEL = "ministral-3:3b-instruct-2512-q4_K_M"'
)

text = text.replace(
    'output_path = project_root / "data" / "raw" / "runs.jsonl"',
    'output_path = project_root / "data" / "raw" / "runs_ministral3_3b_colab.jsonl"'
)

text = text.replace(
    'error_path = project_root / "logs" / "experiment_errors.jsonl"',
    'error_path = project_root / "logs" / "experiment_errors_ministral3_3b_colab.jsonl"'
)

target.write_text(text, encoding="utf-8")

print("Created:", target)

Created: scripts/run_experiment_ministral3_colab.py


In [ ]:
!grep -n 'MODEL =' scripts/run_experiment_ministral3_colab.py

11:MODEL = "ministral-3:3b-instruct-2512-q4_K_M"


In [ ]:
!python scripts/run_experiment_ministral3_colab.py --limit 1

Model: ministral-3:3b-instruct-2512-q4_K_M
Tasks in scope: 1
Expected runs: 2
Already completed: 0
Remaining: 2
Output: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_ministral3_3b_colab.jsonl

Warming up the model...
Warm-up complete.

[1/1] P0001 - baseline ... done | in=638 out=227 total=865 time=3.79s
[1/1] P0001 - optimized ... done | in=607 out=207 total=814 time=3.49s

Run pass finished.
Completed: 2/2
Still missing: 0

All experimental runs completed successfully.


In [ ]:
from pathlib import Path

for p in [
    Path("data/raw/runs_ministral3_3b_colab.jsonl"),
    Path("logs/experiment_errors_ministral3_3b_colab.jsonl"),
]:
    if p.exists():
        p.unlink()
        print("Deleted pilot file:", p)

Deleted pilot file: data/raw/runs_ministral3_3b_colab.jsonl


In [ ]:
!python scripts/run_experiment_ministral3_colab.py

Model: ministral-3:3b-instruct-2512-q4_K_M
Tasks in scope: 100
Expected runs: 200
Already completed: 0
Remaining: 200
Output: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_ministral3_3b_colab.jsonl

Warming up the model...
Warm-up complete.

[1/100] P0001 - baseline ... done | in=638 out=227 total=865 time=3.86s
[1/100] P0001 - optimized ... done | in=607 out=207 total=814 time=3.58s
[2/100] P0002 - optimized ... done | in=618 out=1308 total=1926 time=23.99s
[2/100] P0002 - baseline ... done | in=631 out=1428 total=2059 time=23.28s
[3/100] P0003 - baseline ... done | in=633 out=389 total=1022 time=6.42s
[3/100] P0003 - optimized ... done | in=612 out=354 total=966 time=5.57s
[4/100] P0004 - optimized ... done | in=601 out=273 total=874 time=4.27s
[4/100] P0004 - baseline ... done | in=619 out=246 total=865 time=3.97s
[5/100] P0005 - baseline ... done | in=618 out=280 total=898 time=4.55s
[5/100] P0005 - optimized ... done | in=606 out=187 total=793 time=3.03s
[6/100] P0

In [ ]:
import json
from pathlib import Path

result_file = Path("data/raw/runs_ministral3_3b_colab.jsonl")

records = [
    json.loads(line)
    for line in result_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("Total records:", len(records))

Total records: 200


In [ ]:
from collections import Counter

print("Total records:", len(records))
print("Variants:", Counter(r["variant"] for r in records))
print("Settings:", Counter((r["model"], r["temperature"], r["seed"]) for r in records))

keys = [(r["prompt_id"], r["variant"]) for r in records]
print("Unique prompt/variant records:", len(set(keys)))

Total records: 200
Variants: Counter({'baseline': 100, 'optimized': 100})
Settings: Counter({('ministral-3:3b-instruct-2512-q4_K_M', 0, 42): 200})
Unique prompt/variant records: 200


In [ ]:
from collections import defaultdict

pairs = defaultdict(dict)

for r in records:
    pairs[r["prompt_id"]][r["variant"]] = r

baseline_input = sum(p["baseline"]["input_tokens"] for p in pairs.values())
optimized_input = sum(p["optimized"]["input_tokens"] for p in pairs.values())

baseline_output = sum(p["baseline"]["output_tokens"] for p in pairs.values())
optimized_output = sum(p["optimized"]["output_tokens"] for p in pairs.values())

baseline_total = sum(p["baseline"]["total_tokens"] for p in pairs.values())
optimized_total = sum(p["optimized"]["total_tokens"] for p in pairs.values())

baseline_time = sum(p["baseline"]["wall_time_s"] for p in pairs.values())
optimized_time = sum(p["optimized"]["wall_time_s"] for p in pairs.values())

def reduction(b, o):
    return (b - o) / b * 100

fewer_total = sum(
    p["optimized"]["total_tokens"] < p["baseline"]["total_tokens"]
    for p in pairs.values()
)

faster = sum(
    p["optimized"]["wall_time_s"] < p["baseline"]["wall_time_s"]
    for p in pairs.values()
)

print("=== MINISTRAL 3B SUMMARY ===")
print(f"Input tokens:  baseline={baseline_input:,}  optimized={optimized_input:,}  reduction={reduction(baseline_input, optimized_input):.2f}%")
print(f"Output tokens: baseline={baseline_output:,}  optimized={optimized_output:,}  reduction={reduction(baseline_output, optimized_output):.2f}%")
print(f"Total tokens:  baseline={baseline_total:,}  optimized={optimized_total:,}  reduction={reduction(baseline_total, optimized_total):.2f}%")
print(f"Wall time:     baseline={baseline_time:.2f}s  optimized={optimized_time:.2f}s  reduction={reduction(baseline_time, optimized_time):.2f}%")
print(f"Optimized used fewer total tokens in: {fewer_total}/100 pairs")
print(f"Optimized was faster in: {faster}/100 pairs")

=== MINISTRAL 3B SUMMARY ===
Input tokens:  baseline=78,948  optimized=65,168  reduction=17.45%
Output tokens: baseline=84,501  optimized=70,239  reduction=16.88%
Total tokens:  baseline=163,449  optimized=135,407  reduction=17.16%
Wall time:     baseline=1421.21s  optimized=1174.99s  reduction=17.32%
Optimized used fewer total tokens in: 70/100 pairs
Optimized was faster in: 63/100 pairs


In [ ]:
from collections import Counter

print(Counter(r.get("done_reason") for r in records))

Counter({'stop': 200})
